# northeast-region-cleaner for Massachusetts Crash Map
- See README in [https://github.com/picturedigits/mass-crash-map](https://github.com/picturedigits/mass-crash-map)
- Expected Runtime: 2-5 minutes

# Importing all libraries

In [1]:
!pip install pandas
!pip install numpy
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Collecting MassDOT crash data for specific region and years (update in 2027)

In [2]:
all_features = []

#Inserted years individually because "2023v" naming issue in MassDOT crash server
#https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT

years = ('2021','2022','2023v','2024','2025','2026')

#Regional Planning Area (RPA) 
#northeast = ['NMCOG','MVPC']

regions = ['NMCOG','MVPC']
region_str = ", ".join([f"'{c}'" for c in regions])

for year in years:
   
    base_url = f"https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT/MASSDOT_ODP_OPEN_{year}/FeatureServer/0/query"

    
    params = {
        "where": f"RPA_ABBR IN ({region_str})",
        "outFields": "*",
        "outSR": "4326",
        "f": "json",
        "returnGeometry": "true",
        "resultOffset": 0,
        "resultRecordCount": 2000
    }

    while True:
        
        response = requests.get(base_url, params=params)
        data = response.json()
        features = data.get("features", [])
        
        if not features:
            break
        
        all_features.extend(features)
        params["resultOffset"] += params["resultRecordCount"]

records = [f["attributes"] for f in all_features]

df = pd.DataFrame(records)
print("Shape:", df.shape)
df.head()

Shape: (67254, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATE_TEXT,CRASH_TIME_2,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,4915219,METHUEN,01 01 2021,11:08 PM,1.609542e+12,11:00PM to 11:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Minor Arterial,4616345,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4915218,METHUEN,01 01 2021,3:15 AM,1.609471e+12,03:00AM to 03:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),1,...,NaN,Major Collector,4616346,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4915019,HAVERHILL,01 02 2021,11:00 AM,1.609585e+12,11:00AM to 11:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Minor Arterial,4616515,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4914973,HAVERHILL,01 02 2021,7:00 PM,1.609614e+12,07:00PM to 07:59PM,Closed,Non-fatal injury,Suspected Serious Injury (A),2,...,NaN,Minor Arterial,4616561,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4914972,HAVERHILL,01 02 2021,9:40 AM,1.609580e+12,09:00AM to 09:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Local,4616562,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#CRASH_TIME_2 & CRASH_DATE_TEXT columns are of type str
#LAT & LON are of type float64
#The following code ensures time and date are of type datetime and lat and lon are numeric
df["CRASH_TIME"] = pd.to_datetime(df["CRASH_TIME_2"], format="%I:%M %p", errors="coerce").dt.time
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE_TEXT"], errors="coerce")
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df = df.drop(columns=["CRASH_DATE_TEXT", "CRASH_TIME_2"])
mass_crashes = df
print("Shape of MASSDOT dataset:", mass_crashes.shape)
mass_crashes.head()

Shape of MASSDOT dataset: (67254, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,NUMB_NONFATAL_INJR,NUMB_FATAL_INJR,...,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL,CRASH_TIME,CRASH_DATE
0,4915219,METHUEN,1.609542e+12,11:00PM to 11:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4616345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23:08:00,2021-01-01
1,4915218,METHUEN,1.609471e+12,03:00AM to 03:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),1,0,0,...,4616346,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03:15:00,2021-01-01
2,4915019,HAVERHILL,1.609585e+12,11:00AM to 11:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4616515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11:00:00,2021-01-02
3,4914973,HAVERHILL,1.609614e+12,07:00PM to 07:59PM,Closed,Non-fatal injury,Suspected Serious Injury (A),2,4,0,...,4616561,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19:00:00,2021-01-02
4,4914972,HAVERHILL,1.609580e+12,09:00AM to 09:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4616562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,09:40:00,2021-01-02


# Standardizing Different Types of Vulnerable Roadway Users

In [4]:
col1 = mass_crashes["NON_MTRST_TYPE_CL"]
col2 = mass_crashes["MOST_HRMFL_EVT_CL"]

mass_crashes["PEDESTRIAN"] = np.where(
    col1.str.contains("Pedestrian|Electric Personal Assistive Mobility Device|Wheelchair|Responder|Worker", case=False, na=False) |
    col2.str.contains("Pedestrian", case=False, na=False),
    1,
    0
)

#contains Bicyclist in column1 or Cyclist in column2 = 1, but if "Motorized" = false
mass_crashes["CYCLIST"] = np.where(
    (
        col1.str.contains("Bicyclist|Cyclist|Skater|Non-Motorized Scooter Rider|Micromobility|Skateboarder|Tricyclist", case=False, na=False) |
        col2.str.contains("Cyclist", case=False, na=False)
    )
    &
    ~(
        col1.str.contains("Motorized", case=False, na=False) |
        col2.str.contains("Motorized", case=False, na=False)
    ),
    1,
    0
)

#contains Motorized Bicyclist or Motorized Scooter or moped or Other types below
mass_crashes["OTHER"] = np.where(
    mass_crashes["NON_MTRST_TYPE_CL"].str.contains("Other|Motorized Bicyclist|Motorized Scooter Rider|Passenger|Farm|Unknown", case=False, na=False) |
    mass_crashes["MOST_HRMFL_EVT_CL"].str.contains("Other Vulnerable|moped", case=False, na=False),
    1,
    0
)

mass_crashes['SEVERITY'] = mass_crashes['CRASH_SEVERITY_DESCR'].map({
    'Fatal injury': 1,
    'Non-fatal injury': 2
}).fillna(0)


mass_crashes['INTERSTATE'] = np.where(
    mass_crashes['F_CLASS'].str.contains("Interstate", case=False, na=False),
    1,
    0
)

#trims "Local police" to "Local", "State police" to "State", etc
mass_crashes['POLICE'] = mass_crashes['POLC_AGNCY_TYPE_DESCR'].str.split().str[0]
mass_crashes['ID'] = mass_crashes['CRASH_NUMB']
mass_crashes['MUNI'] = mass_crashes['CITY_TOWN_NAME']
mass_crashes = mass_crashes.drop(columns=["CITY_TOWN_NAME", "CRASH_NUMB", "POLC_AGNCY_TYPE_DESCR"])
mass_crashes['SOURCE'] = 'MassDOT'
mass_crashes['YEAR'] = pd.to_datetime(mass_crashes["CRASH_DATE"]).dt.year

In [5]:
cols_to_move = [
    "SOURCE",
    
    "ID",

    "MUNI",

    "YEAR",

    "CRASH_DATE",

    "SEVERITY",

    "CRASH_TIME",

    "POLICE",

    "PEDESTRIAN",

    "CYCLIST",

    "OTHER",

    "LAT",

    "LON",

    "INTERSTATE",

    
]
df = mass_crashes[cols_to_move + [c for c in mass_crashes.columns if c not in cols_to_move]]
df.head()

,SOURCE,ID,MUNI,YEAR,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,PEDESTRIAN,CYCLIST,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,MassDOT,4915219,METHUEN,2021,2021-01-01,0.0,23:08:00,Local,0,0,...,NaN,Minor Arterial,4616345,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MassDOT,4915218,METHUEN,2021,2021-01-01,0.0,03:15:00,Local,0,0,...,NaN,Major Collector,4616346,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MassDOT,4915019,HAVERHILL,2021,2021-01-02,0.0,11:00:00,Local,0,0,...,NaN,Minor Arterial,4616515,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MassDOT,4914973,HAVERHILL,2021,2021-01-02,2.0,19:00:00,Local,0,0,...,NaN,Minor Arterial,4616561,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MassDOT,4914972,HAVERHILL,2021,2021-01-02,0.0,09:40:00,Local,0,0,...,NaN,Local,4616562,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# UPDATE INDEX AS NEEDED: We keep only relevant columns and drop rows that don't have a date
df_short = df.iloc[:, :14]
df_short = df_short.dropna(subset=["CRASH_DATE"])

In [7]:
# We make sure numerical data is of type int or float, and strings of type str
df_short["CRASH_DATE"] = pd.to_datetime(df_short["CRASH_DATE"]).dt.date
df_short['SEVERITY'] = pd.to_numeric(df_short['SEVERITY'], errors='coerce').astype('Int64')
df_short['INTERSTATE'] = df_short['INTERSTATE'].astype(str)
df_short['ID'] = df_short['ID'].astype(str)

In [8]:
df_short.info()

<class 'pandas.DataFrame'>
RangeIndex: 67254 entries, 0 to 67253
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SOURCE      67254 non-null  str    
 1   ID          67254 non-null  str    
 2   MUNI        67254 non-null  str    
 3   YEAR        67254 non-null  int32  
 4   CRASH_DATE  67254 non-null  object 
 5   SEVERITY    67254 non-null  Int64  
 6   CRASH_TIME  67069 non-null  object 
 7   POLICE      67251 non-null  object 
 8   PEDESTRIAN  67254 non-null  int64  
 9   CYCLIST     67254 non-null  int64  
 10  OTHER       67254 non-null  int64  
 11  LAT         64531 non-null  float64
 12  LON         64531 non-null  float64
 13  INTERSTATE  67254 non-null  str    
dtypes: Int64(1), float64(2), int32(1), int64(3), object(3), str(4)
memory usage: 7.0+ MB


In [9]:
df_short.head()

,SOURCE,ID,MUNI,YEAR,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,PEDESTRIAN,CYCLIST,OTHER,LAT,LON,INTERSTATE
0,MassDOT,4915219,METHUEN,2021,2021-01-01,0,23:08:00,Local,0,0,0,42.702818,-71.211763,0
1,MassDOT,4915218,METHUEN,2021,2021-01-01,0,03:15:00,Local,0,0,0,42.725004,-71.166777,0
2,MassDOT,4915019,HAVERHILL,2021,2021-01-02,0,11:00:00,Local,0,0,0,42.772673,-71.085900,0
3,MassDOT,4914973,HAVERHILL,2021,2021-01-02,2,19:00:00,Local,0,0,0,42.777782,-71.085150,0
4,MassDOT,4914972,HAVERHILL,2021,2021-01-02,0,09:40:00,Local,0,0,0,42.756487,-71.090926,0


### Renaming columns to align with Mass Crash Map format

In [10]:
renaming = {
    "LAT": "lat",
    "LON": "lng",
    "CRASH_DATE": "date",
    "CRASH_TIME": "time",
    "YEAR": "year",
    "PEDESTRIAN": "pedestrian",
    "CYCLIST": "cyclist",
    "OTHER": "other",
    "INTERSTATE": "interstate",
    "SEVERITY": "severity",
    "ID": "id",
    "MUNI": "muni",
    "SOURCE": "source",
    "POLICE": "police"
}

df = df_short.rename(columns= renaming)

In [11]:
# rewrite file name to match above
df.to_csv("northeast.csv")